In [54]:
# =========================================================
# CELLULE 1 — IMPORTS
# =========================================================

import pandas as pd
import psycopg2
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_style("whitegrid")

In [55]:
# =========================================================
# CELLULE 2 — CONNEXION POSTGRESQL
# =========================================================

DB_CONFIG = {
    "user": "postgres",
    "password": "admin",
    "host": "localhost",
    "port": "5432"
}

DB_NAME = "metals_db"

conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    port=DB_CONFIG["port"]
)

print("Connexion PostgreSQL réussie.")

Connexion PostgreSQL réussie.


In [56]:
# =========================================================
# CELLULE 3 — LISTE DES TABLES
# =========================================================

query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
"""

tables_df = pd.read_sql(query, conn)

display(tables_df)

C:\Users\ibenl\AppData\Local\Temp\ipykernel_18276\2372545455.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables_df = pd.read_sql(query, conn)


,table_name
0,raw_data
1,gdelt_data
2,vix_oil_data
3,cleaned_data
4,reserves_gold
5,Macroeconomic_data
6,dim_date


In [57]:
# =========================================================
# CELLULE 4 — CHARGEMENT DES TABLES
# =========================================================

data = {}

for table in tables_df["table_name"]:

    try:

        print(f"Loading table: {table}")

        df = pd.read_sql(
            f'SELECT * FROM public."{table}"',
            conn
        )

        data[table] = df

        print(f"{table} loaded with shape {df.shape}")

    except Exception as e:

        print(f"Error loading {table}: {e}")

Loading table: raw_data


C:\Users\ibenl\AppData\Local\Temp\ipykernel_18276\1148487819.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


raw_data loaded with shape (58272, 10)
Loading table: gdelt_data


KeyboardInterrupt: 

In [ ]:
# =========================================================
# CELLULE 5 — DATAFRAMES PRINCIPAUX
# =========================================================

macro_df = data.get("Macroeconomic_data")

prices_df = data.get("cleaned_data")

geopo_df = data.get("gdelt_data")

gold_reser_df = data.get("reserves_gold")

vix_oil_df = data.get("vix_oil_data")

In [ ]:
# =========================================================
# CELLULE 6 — DATASETS
# =========================================================

datasets = {
    "prices_df": prices_df,
    "macro_df": macro_df,
    "geopo_df": geopo_df,
    "gold_reser_df": gold_reser_df,
    "vix_oil_df": vix_oil_df
}

print("Datasets enregistrés :", list(datasets.keys()))

Datasets enregistrés : ['prices_df', 'macro_df', 'geopo_df', 'gold_reser_df', 'vix_oil_df']


In [ ]:
resultat = prices_df[prices_df["date"] == "2018-01-01"]
print(resultat)

       metals           Pays  Année        date   gold_24k   gold_22k  \
260      gold        algerie   2018  2018-01-01   4821.910   4420.080   
2688     gold       belgique   2018  2018-01-01     34.982     32.067   
5116     gold        tunisie   2018  2018-01-01    103.170     94.573   
7544     gold         canada   2018  2018-01-01     52.758     48.362   
9972     gold         france   2018  2018-01-01     34.982     32.067   
12400    gold       Pays-bas   2018  2018-01-01     34.982     32.067   
14828    gold    royaume-uni   2018  2018-01-01     31.100     28.508   
17256    gold       cameroun   2018  2018-01-01  22947.000  21034.000   
19684    gold  cote-d’ivoire   2018  2018-01-01  22947.000  21034.000   
22112    gold     etats-unis   2018  2018-01-01     42.016     38.515   
24540    gold          maroc   2018  2018-01-01    392.120    359.440   
26968    gold         suisse   2018  2018-01-01     40.957     37.544   
29396  silver        algerie   2018  2018-01-01    

In [ ]:
prices_df.columns

Index(['metals', 'Pays', 'Année', 'date', 'gold_24k', 'gold_22k', 'gold_18k',
       'gold_14k', 'gold_10k', 'silver_price', 'devise'],
      dtype='object')

In [ ]:

gold_df = prices_df[prices_df["metals"] == "gold"][
    ["Pays", "date", "gold_24k" , "gold_22k", "gold_18k",  "gold_14k" , "gold_10k"]
]

# Séparer silver
silver_df = prices_df[prices_df["metals"] == "silver"][
    ["Pays", "date", "silver_price"]
]

# Fusionner
resultat = pd.merge(
    gold_df,
    silver_df,
    on=["Pays", "date"],
    how="inner"
)

print(resultat)

          Pays        date  gold_24k  gold_22k  gold_18k  gold_14k  gold_10k  \
0      algerie  2017-01-02   4095.46   3754.18  3071.600  2389.020  1706.440   
1      algerie  2017-01-03   4124.52   3780.81  3093.390  2405.970  1718.550   
2      algerie  2017-01-04   4143.24   3797.97  3107.430  2416.890  1726.350   
3      algerie  2017-01-05   4183.85   3835.20  3137.890  2440.580  1743.270   
4      algerie  2017-01-06   4161.20   3814.43  3120.900  2427.360  1733.830   
...        ...         ...       ...       ...       ...       ...       ...   
29131   suisse  2026-04-20    120.84    110.77    90.627    70.488    50.348   
29132   suisse  2026-04-21    118.43    108.56    88.825    69.086    49.347   
29133   suisse  2026-04-22    119.42    109.46    89.562    69.659    49.757   
29134   suisse  2026-04-23    118.64    108.75    88.978    69.205    49.432   
29135   suisse  2026-04-24    118.86    108.95    89.142    69.332    49.523   

       silver_price  
0           56.88

In [ ]:
resultat.head()

,Pays,date,gold_24k,gold_22k,gold_18k,gold_14k,gold_10k,silver_price
0,algerie,2017-01-02,4095.46,3754.18,3071.60,2389.02,1706.44,56.885
1,algerie,2017-01-03,4124.52,3780.81,3093.39,2405.97,1718.55,58.082
2,algerie,2017-01-04,4143.24,3797.97,3107.43,2416.89,1726.35,58.624
3,algerie,2017-01-05,4183.85,3835.20,3137.89,2440.58,1743.27,58.856
4,algerie,2017-01-06,4161.20,3814.43,3120.90,2427.36,1733.83,58.545


In [ ]:
resultat.tail()

,Pays,date,gold_24k,gold_22k,gold_18k,gold_14k,gold_10k,silver_price
29131,suisse,2026-04-20,120.84,110.77,90.627,70.488,50.348,2.0031
29132,suisse,2026-04-21,118.43,108.56,88.825,69.086,49.347,1.9297
29133,suisse,2026-04-22,119.42,109.46,89.562,69.659,49.757,1.9576
29134,suisse,2026-04-23,118.64,108.75,88.978,69.205,49.432,1.9072
29135,suisse,2026-04-24,118.86,108.95,89.142,69.332,49.523,1.9098


In [ ]:
macro_df = macro_df[macro_df["date"] >= "2016-01-01"]
print(macro_df.shape)

(5452, 7)


In [ ]:
macro_df.head()

,date,fed_rate,real_rate,CPI,GDP,DXY,Unemployment
3361,2016-01-01,0.34,0.734506,237.652,18525.933,NaN,4.8
3362,2016-01-04,NaN,NaN,NaN,NaN,114.1595,NaN
3363,2016-01-05,NaN,NaN,NaN,NaN,114.2649,NaN
3364,2016-01-06,NaN,NaN,NaN,NaN,114.6177,NaN
3365,2016-01-07,NaN,NaN,NaN,NaN,114.6516,NaN


In [ ]:
macro_df.tail()

,date,fed_rate,real_rate,CPI,GDP,DXY,Unemployment
12161,2026-04-20,NaN,NaN,NaN,NaN,118.2374,NaN
12162,2026-04-21,NaN,NaN,NaN,NaN,118.4331,NaN
12163,2026-04-22,NaN,NaN,NaN,NaN,118.6004,NaN
12164,2026-04-23,NaN,NaN,NaN,NaN,118.7155,NaN
12165,2026-04-24,NaN,NaN,NaN,NaN,118.7294,NaN


In [59]:
macro_df.isnull().sum()

date               0
fed_rate        5204
real_rate       5204
CPI             5208
GDP             5370
DXY              310
Unemployment    5208
dtype: int64

In [60]:
(macro_df.isnull().mean() * 100).sort_values(ascending=False)

GDP             98.495965
CPI             95.524578
Unemployment    95.524578
fed_rate        95.451211
real_rate       95.451211
DXY              5.685987
date             0.000000
dtype: float64

In [62]:
macro_cols = ["fed_rate", "real_rate", "CPI", "GDP", "DXY", "Unemployment"]

macro_df[macro_cols] = macro_df[macro_cols].ffill()

In [63]:
macro_df.isnull().sum()

date            0
fed_rate        0
real_rate       0
CPI             0
GDP             0
DXY             1
Unemployment    0
dtype: int64

In [71]:
macro_df.tail()

,date,fed_rate,real_rate,CPI,GDP,DXY,Unemployment
12161,2026-04-20,3.64,1.584927,330.293,31856.257,118.2374,4.3
12162,2026-04-21,3.64,1.584927,330.293,31856.257,118.4331,4.3
12163,2026-04-22,3.64,1.584927,330.293,31856.257,118.6004,4.3
12164,2026-04-23,3.64,1.584927,330.293,31856.257,118.7155,4.3
12165,2026-04-24,3.64,1.584927,330.293,31856.257,118.7294,4.3


In [65]:
geopo_df.head(
)

,date,country,total_events,political_events,war_intensity,crisis_index,political_pressure
0,1920-01-01,AFG,423,203,0.321513,-2.416548,0.479905
1,1920-01-01,AFR,463,118,0.086393,0.481857,0.254860
2,1920-01-01,AGO,37,19,0.000000,-1.835135,0.513514
3,1920-01-01,ALB,29,10,0.000000,0.351724,0.344828
4,1920-01-01,ARE,219,34,0.041096,1.403196,0.155251


In [66]:
geopo_df = geopo_df[geopo_df["date"] >= "2016-01-01"]
print(geopo_df.shape)

(649765, 7)


In [72]:
geopo_df.head()

,date,country,total_events,political_events,war_intensity,crisis_index,political_pressure
1175646,2016-01-01,AFG,1100,408,0.214545,-1.031818,0.370909
1175647,2016-01-01,AFR,528,139,0.068182,0.721402,0.263258
1175648,2016-01-01,ALB,17,6,0.000000,-2.000000,0.352941
1175649,2016-01-01,ARE,363,74,0.063361,0.633333,0.203857
1175650,2016-01-01,ARG,70,11,0.014286,0.987143,0.157143


In [73]:
geopo_df.isnull().sum()

date                  0
country               0
total_events          0
political_events      0
war_intensity         0
crisis_index          0
political_pressure    0
dtype: int64

In [79]:
gold_reser_df.head(100)


,country_name,country_code,year,value
0,Tunisia,TUN,2010,9764265037
1,United Kingdom,GBR,2010,98025460114
2,United States,USA,2010,488928508742
3,Morocco,MAR,2010,23709466593
4,Cameroon,CMR,2010,3642642765
5,Canada,CAN,2010,57151157771
6,France,FRA,2010,165852088071
7,Tunisia,TUN,2011,7785305507
8,United Kingdom,GBR,2011,109733899591
9,United States,USA,2011,537267041107


In [78]:
gold_reser_df.shape

(105, 4)

In [80]:
gold_reser_df["reserve_yoy"] = (
    gold_reser_df.groupby("country_code")["value"]
    .pct_change()
)

In [81]:
gold_reser_df.head(10)

,country_name,country_code,year,value,reserve_yoy
0,Tunisia,TUN,2010,9764265037,NaN
1,United Kingdom,GBR,2010,98025460114,NaN
2,United States,USA,2010,488928508742,NaN
3,Morocco,MAR,2010,23709466593,NaN
4,Cameroon,CMR,2010,3642642765,NaN
5,Canada,CAN,2010,57151157771,NaN
6,France,FRA,2010,165852088071,NaN
7,Tunisia,TUN,2011,7785305507,-0.202674
8,United Kingdom,GBR,2011,109733899591,0.119443
9,United States,USA,2011,537267041107,0.098866


In [82]:
gold_reser_df["reserve_trend"] = (
    gold_reser_df.groupby("country_code")["reserve_yoy"]
    .rolling(3)
    .mean()
    .reset_index(0, drop=True)
)

In [83]:
gold_reser_df.head(10)

,country_name,country_code,year,value,reserve_yoy,reserve_trend
0,Tunisia,TUN,2010,9764265037,NaN,NaN
1,United Kingdom,GBR,2010,98025460114,NaN,NaN
2,United States,USA,2010,488928508742,NaN,NaN
3,Morocco,MAR,2010,23709466593,NaN,NaN
4,Cameroon,CMR,2010,3642642765,NaN,NaN
5,Canada,CAN,2010,57151157771,NaN,NaN
6,France,FRA,2010,165852088071,NaN,NaN
7,Tunisia,TUN,2011,7785305507,-0.202674,NaN
8,United Kingdom,GBR,2011,109733899591,0.119443,NaN
9,United States,USA,2011,537267041107,0.098866,NaN


In [84]:
global_gold_demand = (
    gold_reser_df.groupby("year")["reserve_yoy"]
    .mean()
)

In [86]:
gold_reser_df["date"] = pd.to_datetime(
    gold_reser_df["year"].astype(str) + "-01-01"
)

gold_reser_daily = (
    gold_reser_df
    .set_index("date")
    .groupby("country_code")
    .resample("D")
    .ffill()
)

C:\Users\ibenl\AppData\Local\Temp\ipykernel_18276\4258972823.py:10: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .ffill()


In [87]:
vix_oil_df.head()

,Date,vix,oil
0,2016-01-04,20.700001,36.759998
1,2016-01-05,19.340000,35.970001
2,2016-01-06,20.590000,33.970001
3,2016-01-07,24.990000,33.270000
4,2016-01-08,27.010000,33.160000


In [88]:
vix_oil_df.tail()

,Date,vix,oil
2594,2026-04-27,18.020000,96.370003
2595,2026-04-28,17.830000,99.930000
2596,2026-04-29,18.809999,106.879997
2597,2026-04-30,16.889999,105.070000
2598,2026-05-01,16.990000,101.940002


In [89]:
vix_oil_df.isnull().sum()

Date    0
vix     2
oil     2
dtype: int64

In [90]:
vix_oil_df[vix_oil_df[['vix', 'oil']].isnull().any(axis=1)]

,Date,vix,oil
194,2016-10-10,13.38,NaN
218,2016-11-11,14.17,NaN
2269,2025-01-09,NaN,73.919998
2390,2025-07-04,NaN,66.500000


In [91]:
vix_oil_df[['vix', 'oil']] = vix_oil_df[['vix', 'oil']].ffill()

In [92]:
vix_oil_df.isnull().sum()

Date    0
vix     0
oil     0
dtype: int64

In [94]:
vix_oil_df.loc[[194, 193, 217, 218, 2269, 2268, 2390, 2389], ['vix', 'oil']]

,vix,oil
194,13.380000,49.810001
193,13.480000,49.810001
217,14.740000,44.660000
218,14.170000,44.660000
2269,17.700001,73.919998
2268,17.700001,73.320000
2390,16.379999,66.500000
2389,16.379999,67.000000


In [97]:
def statistical_info(df):
    print("🔎 DATAFRAME OVERVIEW")
    print("-" * 50)

    # Shape
    print(f"📌 Shape (rows, columns): {df.shape}")
    print("-" * 50)

    # Data types
    print("📌 Data types:")
    print(df.dtypes)
    print("-" * 50)

    # Missing values (count)
    print("📌 Missing values (count):")
    print(df.isnull().sum())
    print("-" * 50)

    # Basic statistics for numerical columns
    print("📌 Descriptive statistics (numerical):")
    print(df.describe().T)
    print("-" * 50)

    # Optional: duplicates
    print(f"📌 Duplicate rows: {df.duplicated().sum()}")
    print("-" * 50)

    # Column list
    print("📌 Columns:")
    print(list(df.columns))
    print("-" * 50)

In [98]:
statistical_info(prices_df)

🔎 DATAFRAME OVERVIEW
--------------------------------------------------
📌 Shape (rows, columns): (58272, 11)
--------------------------------------------------
📌 Data types:
metals                  object
Pays                    object
Année                    int64
date            datetime64[ns]
gold_24k               float64
gold_22k               float64
gold_18k               float64
gold_14k               float64
gold_10k               float64
silver_price           float64
devise                  object
dtype: object
--------------------------------------------------
📌 Missing values (count):
metals              0
Pays                0
Année               0
date                0
gold_24k            0
gold_22k            0
gold_18k            0
gold_14k            0
gold_10k            0
silver_price        0
devise          29136
dtype: int64
--------------------------------------------------
📌 Descriptive statistics (numerical):
                count                           me

In [99]:
statistical_info(macro_df)

🔎 DATAFRAME OVERVIEW
--------------------------------------------------
📌 Shape (rows, columns): (5452, 7)
--------------------------------------------------
📌 Data types:
date            datetime64[ns]
fed_rate               float64
real_rate              float64
CPI                    float64
GDP                    float64
DXY                    float64
Unemployment           float64
dtype: object
--------------------------------------------------
📌 Missing values (count):
date            0
fed_rate        0
real_rate       0
CPI             0
GDP             0
DXY             1
Unemployment    0
dtype: int64
--------------------------------------------------
📌 Descriptive statistics (numerical):
               count                           mean                  min  \
date            5452  2021-02-27 21:51:06.471019776  2016-01-01 00:00:00   
fed_rate      5452.0                       2.238648                 0.05   
real_rate     5452.0                       0.917001            -

In [100]:
statistical_info(vix_oil_df)

🔎 DATAFRAME OVERVIEW
--------------------------------------------------
📌 Shape (rows, columns): (2599, 3)
--------------------------------------------------
📌 Data types:
Date    datetime64[ns]
vix            float64
oil            float64
dtype: object
--------------------------------------------------
📌 Missing values (count):
Date    0
vix     0
oil     0
dtype: int64
--------------------------------------------------
📌 Descriptive statistics (numerical):
       count                           mean                  min  \
Date    2599  2021-03-01 22:15:50.211619840  2016-01-04 00:00:00   
vix   2599.0                      18.553005                 9.14   
oil   2599.0                      64.096533           -37.630001   

                      25%                  50%                  75%  \
Date  2018-08-01 12:00:00  2021-03-03 00:00:00  2023-09-30 12:00:00   
vix                  13.6            16.860001            21.554999   
oil             51.690001            63.580002    

In [101]:
statistical_info(geopo_df)

🔎 DATAFRAME OVERVIEW
--------------------------------------------------
📌 Shape (rows, columns): (649765, 7)
--------------------------------------------------
📌 Data types:
date                  datetime64[ns]
country                       object
total_events                   int64
political_events               int64
war_intensity                float64
crisis_index                 float64
political_pressure           float64
dtype: object
--------------------------------------------------
📌 Missing values (count):
date                  0
country               0
total_events          0
political_events      0
war_intensity         0
crisis_index          0
political_pressure    0
dtype: int64
--------------------------------------------------
📌 Descriptive statistics (numerical):
                       count                           mean  \
date                  649765  2021-01-21 21:24:41.381730048   
total_events        649765.0                     448.895582   
political_events 

In [108]:
country_to_currency = {
    "algerie": "DZD",          # Dinar algérien
    "belgique": "EUR_bel",         # Euro
    "tunisie": "TND",          # Dinar tunisien
    "canada": "CAD",          # Dollar canadien
    "france": "EUR_fr",          # Euro
    "Pays-bas": "EUR_pb",        # Euro
    "royaume-uni": "GBP",     # Livre sterling
    "cameroun": "XAF",        # Franc CFA
    "cote-d’ivoire": "XOF",   # Franc CFA BCEAO
    "etats-unis": "USD",     # Dollar US
    "maroc": "MAD",          # Dirham marocain
    "suisse": "CHF"          # Franc suisse
}

In [102]:
prices_df.head()

,metals,Pays,Année,date,gold_24k,gold_22k,gold_18k,gold_14k,gold_10k,silver_price,devise
0,gold,algerie,2017,2017-01-02,4095.46,3754.18,3071.60,2389.02,1706.44,0.0,DA
1,gold,algerie,2017,2017-01-03,4124.52,3780.81,3093.39,2405.97,1718.55,0.0,DA
2,gold,algerie,2017,2017-01-04,4143.24,3797.97,3107.43,2416.89,1726.35,0.0,DA
3,gold,algerie,2017,2017-01-05,4183.85,3835.20,3137.89,2440.58,1743.27,0.0,DA
4,gold,algerie,2017,2017-01-06,4161.20,3814.43,3120.90,2427.36,1733.83,0.0,DA


In [109]:
prices_df['devise'] = prices_df['Pays'].map(country_to_currency)

In [110]:
prices_df.isnull().sum()

metals          0
Pays            0
Année           0
date            0
gold_24k        0
gold_22k        0
gold_18k        0
gold_14k        0
gold_10k        0
silver_price    0
devise          0
dtype: int64

In [111]:
prices_df['devise'].value_counts()

devise
DZD        4856
EUR_bel    4856
TND        4856
CAD        4856
EUR_fr     4856
EUR_pb     4856
GBP        4856
XAF        4856
XOF        4856
USD        4856
MAD        4856
CHF        4856
Name: count, dtype: int64